In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
data_path = Path("../data/raw/train.tsv")

df = pd.read_csv(
    data_path,
    sep="\t",
    low_memory=False
)

print("Original dataset shape:", df.shape)

Original dataset shape: (1482535, 8)


In [3]:
women_shoes = df[
    df["category_name"]
    .fillna("")
    .str.startswith("Women/Shoes")
].copy()

print("Women's shoe listings before cleaning:", women_shoes.shape)

Women's shoe listings before cleaning: (77654, 8)


In [4]:
women_shoes["shoe_type"] = (
    women_shoes["category_name"]
    .str.split("/")
    .str[2]
)

women_shoes[["category_name", "shoe_type"]].head()

,category_name,shoe_type
14,Women/Shoes/Boots,Boots
70,Women/Shoes/Athletic,Athletic
107,Women/Shoes/Boots,Boots
108,Women/Shoes/Boots,Boots
114,Women/Shoes/Loafers & Slip-Ons,Loafers & Slip-Ons


In [5]:
selected_categories = [
    "Boots",
    "Sandals",
    "Athletic",
    "Fashion Sneakers",
    "Pumps",
    "Flats",
    "Loafers & Slip-Ons"
]

women_shoes = women_shoes[
    women_shoes["shoe_type"].isin(selected_categories)
].copy()

print("Listings after category selection:", women_shoes.shape)
women_shoes["shoe_type"].value_counts()

Listings after category selection: (73737, 9)


shoe_type
Boots                 18864
Sandals               14662
Athletic              12662
Fashion Sneakers      10164
Pumps                  7454
Flats                  5422
Loafers & Slip-Ons     4509
Name: count, dtype: int64

In [6]:
clean_shoes = women_shoes.dropna(
    subset=[
        "brand_name",
        "price",
        "item_condition_id"
    ]
).copy()

clean_shoes = clean_shoes[
    clean_shoes["price"] > 0
].copy()

print("Listings before cleaning:", len(women_shoes))
print("Listings after cleaning:", len(clean_shoes))
print("Rows removed:", len(women_shoes) - len(clean_shoes))

Listings before cleaning: 73737
Listings after cleaning: 53176
Rows removed: 20561


In [7]:
clean_shoes[
    ["brand_name", "price", "item_condition_id"]
].isna().sum()

brand_name           0
price                0
item_condition_id    0
dtype: int64

In [8]:
(clean_shoes["price"] <= 0).sum()

np.int64(0)

In [9]:
clean_shoes["brand_name"] = (
    clean_shoes["brand_name"]
    .str.strip()
)

clean_shoes["shoe_type"] = (
    clean_shoes["shoe_type"]
    .str.strip()
)

In [10]:
blank_brand_count = (
    clean_shoes["brand_name"]
    .eq("")
    .sum()
)

print("Blank brand names:", blank_brand_count)

Blank brand names: 0


In [11]:
clean_shoes = clean_shoes[
    clean_shoes["brand_name"] != ""
].copy()

In [12]:
print("Final cleaned rows so far:", clean_shoes.shape)

Final cleaned rows so far: (53176, 9)


In [14]:
condition_map = {
    1: "Condition 1",
    2: "Condition 2",
    3: "Condition 3",
    4: "Condition 4",
    5: "Condition 5"
}

clean_shoes["condition_group"] = (
    clean_shoes["item_condition_id"]
    .map(condition_map)
)

clean_shoes[
    ["item_condition_id", "condition_group"]
].drop_duplicates().sort_values("item_condition_id")

,item_condition_id,condition_group
107,1,Condition 1
115,2,Condition 2
14,3,Condition 3
141,4,Condition 4
26049,5,Condition 5


In [15]:
clean_shoes[
    ["item_condition_id", "condition_group"]
].drop_duplicates().sort_values(
    "item_condition_id"
).reset_index(drop=True)

,item_condition_id,condition_group
0,1,Condition 1
1,2,Condition 2
2,3,Condition 3
3,4,Condition 4
4,5,Condition 5


In [18]:
clean_shoes["item_condition_id"].value_counts().sort_index()

item_condition_id
1    11099
2    14296
3    24690
4     2988
5      103
Name: count, dtype: int64

In [19]:
clean_shoes["condition_group"] = clean_shoes["item_condition_id"].astype(str)

In [20]:
clean_shoes["item_condition_id"].replace({
    4: "4–5",
    5: "4–5"
}).value_counts()

item_condition_id
3      24690
2      14296
1      11099
4–5     3091
Name: count, dtype: int64

In [21]:
condition_analysis_map = {
    1: "Condition 1",
    2: "Condition 2",
    3: "Condition 3",
    4: "Condition 4–5",
    5: "Condition 4–5"
}

clean_shoes["condition_group"] = (
    clean_shoes["item_condition_id"]
    .map(condition_analysis_map)
)

clean_shoes["condition_group"].value_counts()

condition_group
Condition 3      24690
Condition 2      14296
Condition 1      11099
Condition 4–5     3091
Name: count, dtype: int64

In [22]:
output_path = Path("../data/processed/clean_womens_shoes.parquet")

clean_shoes.to_parquet(
    output_path,
    index=False
)

print("Saved to:", output_path)
print("Saved rows:", len(clean_shoes))

Saved to: ../data/processed/clean_womens_shoes.parquet
Saved rows: 53176
